# Day 5 — Visualisation with Matplotlib and seaborn
## Master's Course: Quantitative Skills for Biology

> *"A plot is worth a thousand p-values."*

Data visualisation is not just about making figures look nice — it is how we explore data, detect problems, and communicate findings. In genomics, good plots often reveal patterns that tables cannot.

By the end of this session you will be able to:
- Build publication-quality figures with **matplotlib**
- Use **seaborn** for statistical graphics (boxplots, heatmaps)
- Visualise gene expression data: distributions, scatter plots, heatmaps
- Save figures to file

In [ ]:
# Imports — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", font_scale=1.1)  # nicer default style
print("Libraries loaded.")

---
## Setup: recreate the expression dataset from Day 4

We use the same synthetic data so all plots are based on data you already understand.

In [ ]:
np.random.seed(42)

n_genes = 20
gene_names  = [f"GENE_{i:03d}" for i in range(1, n_genes + 1)]
sample_names = [f"ctrl_{i}" for i in range(1, 4)] + [f"treat_{i}" for i in range(1, 4)]
ctrl_cols  = ["ctrl_1",  "ctrl_2",  "ctrl_3"]
treat_cols = ["treat_1", "treat_2", "treat_3"]

# Generate data
ctrl_data  = np.random.normal(8.0, 1.0, (n_genes, 3))
treat_mean = np.full(n_genes, 8.0)
treat_mean[:5] = 10.0
treat_data = np.random.normal(treat_mean[:, None], 1.0, (n_genes, 3))

df = pd.DataFrame(
    np.hstack([ctrl_data, treat_data]),
    index=gene_names,
    columns=sample_names
)
df.index.name = "gene"

# Add summary columns
df["mean_ctrl"]  = df[ctrl_cols].mean(axis=1)
df["mean_treat"] = df[treat_cols].mean(axis=1)
df["log2fc"]     = np.log2((df["mean_treat"] + 0.01) / (df["mean_ctrl"] + 0.01))
df["variance"]   = df[sample_names].var(axis=1)

# Add mock p-values (smaller for genes 1-5)
np.random.seed(7)
df["pvalue"] = np.where(
    df.index.isin([f"GENE_{i:03d}" for i in range(1, 6)]),
    np.random.uniform(0.0001, 0.04, n_genes)[:5],
    np.random.uniform(0.05, 1.0, n_genes)[5:]
)

print(df[sample_names + ["log2fc", "pvalue"]].round(2))

---
## 1. Matplotlib basics — the `fig, ax` pattern

Always use `fig, ax = plt.subplots()` rather than calling `plt.plot()` directly. This gives you full control over each plot element and makes multi-panel figures easy.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

# Plot mean expression per gene in control and treated
ax.plot(gene_names, df["mean_ctrl"],  marker="o", label="Control",  color="steelblue")
ax.plot(gene_names, df["mean_treat"], marker="s", label="Treated",  color="coral")

# Labels and formatting
ax.set_xlabel("Gene")
ax.set_ylabel("Mean expression (log2)")
ax.set_title("Mean expression: Control vs Treated")
ax.tick_params(axis="x", rotation=45)
ax.legend()

plt.tight_layout()
plt.show()

---
## 2. Bar chart — top expressed genes

In [ ]:
# Top 10 genes by overall mean expression
top10 = df.nlargest(10, "mean_ctrl").sort_values("mean_ctrl")

# Colour by regulation status
colors = []
for fc in top10["log2fc"]:
    if fc > 1:    colors.append("tomato")     # upregulated
    elif fc < -1: colors.append("steelblue")  # downregulated
    else:         colors.append("lightgray")  # unchanged

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top10.index, top10["mean_ctrl"], color=colors)
ax.set_xlabel("Mean expression in control (log2)")
ax.set_title("Top 10 expressed genes")

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="tomato",    label="Upregulated (log2FC > 1)"),
    Patch(facecolor="steelblue", label="Downregulated (log2FC < -1)"),
    Patch(facecolor="lightgray", label="Unchanged")
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

---
## 3. Histograms — distributions

Always visualise the distribution of your data before any analysis. Log-transformation is common in genomics to handle the wide dynamic range of expression values.

In [ ]:
# Flatten all values from control and treated samples
ctrl_vals  = df[ctrl_cols].values.flatten()
treat_vals = df[treat_cols].values.flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: distributions on original scale (pretend these are raw counts, not log)
raw_ctrl  = 2 ** ctrl_vals   # convert back to 'raw' counts for illustration
raw_treat = 2 ** treat_vals
axes[0].hist(raw_ctrl,  bins=30, alpha=0.6, color="steelblue", label="Control")
axes[0].hist(raw_treat, bins=30, alpha=0.6, color="coral",     label="Treated")
axes[0].set_title("Raw counts — skewed distribution")
axes[0].set_xlabel("Expression (raw scale)")
axes[0].legend()

# Right: after log2 transformation
axes[1].hist(ctrl_vals,  bins=20, alpha=0.6, color="steelblue", label="Control")
axes[1].hist(treat_vals, bins=20, alpha=0.6, color="coral",     label="Treated")
axes[1].axvline(ctrl_vals.mean(),  color="navy",   linestyle="--", label=f"ctrl mean = {ctrl_vals.mean():.1f}")
axes[1].axvline(treat_vals.mean(), color="darkred", linestyle="--", label=f"treat mean = {treat_vals.mean():.1f}")
axes[1].set_title("After log2 — approximately normal")
axes[1].set_xlabel("Expression (log2 scale)")
axes[1].legend(fontsize=8)

plt.suptitle("Effect of log2 transformation", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Scatter plot — replicate correlation

Before any analysis, check that your biological replicates are correlated. Poor correlation suggests a sample quality problem.

In [ ]:
x = df["ctrl_1"]
y = df["ctrl_2"]

# Pearson correlation
r, p = stats.pearsonr(x, y)

# Colour top-5 variable genes differently
top5_var = df.nlargest(5, "variance").index
colors = ["tomato" if g in top5_var else "steelblue" for g in df.index]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x, y, c=colors, alpha=0.8, edgecolors="white", s=60)

# Annotate the most variable genes
for gene in top5_var[:3]:
    ax.annotate(gene, (df.loc[gene, "ctrl_1"], df.loc[gene, "ctrl_2"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)

# Add diagonal (perfect correlation)
lim = [min(x.min(), y.min()) - 0.5, max(x.max(), y.max()) + 0.5]
ax.plot(lim, lim, "k--", alpha=0.4, label="y = x")
ax.set_xlim(lim); ax.set_ylim(lim)

ax.set_xlabel("ctrl_1 (log2)")
ax.set_ylabel("ctrl_2 (log2)")
ax.set_title(f"Replicate correlation\nr = {r:.3f}, p = {p:.2e}")

plt.tight_layout()
plt.show()

---
## 5. seaborn — statistical graphics

Seaborn builds on matplotlib and produces elegant statistical plots with less code.

In [ ]:
# Melt to long format for seaborn
df_long = df[sample_names].reset_index().melt(
    id_vars="gene", var_name="sample", value_name="expression"
)
df_long["condition"] = df_long["sample"].apply(
    lambda s: "Control" if s.startswith("ctrl") else "Treated"
)
print(df_long.head())

In [ ]:
# Boxplot: expression distribution per sample, coloured by condition
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(
    data=df_long, x="sample", y="expression", hue="condition",
    palette={"Control": "steelblue", "Treated": "coral"},
    dodge=False, ax=ax
)
ax.set_xlabel("")
ax.set_ylabel("Expression (log2)")
ax.set_title("Expression distribution per sample")
plt.tight_layout()
plt.show()

---
## 6. Heatmap — expression across genes and samples

A heatmap shows expression levels for many genes at once. We use **z-score normalisation** (per gene, across samples) so we compare *relative* changes rather than absolute levels.

In [ ]:
# Select the 15 most variable genes
top15 = df.nlargest(15, "variance").index
heat_data = df.loc[top15, sample_names]

# Z-score normalise across samples (per gene)
heat_z = heat_data.apply(stats.zscore, axis=1)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    heat_z,
    cmap="RdBu_r",
    vmin=-2, vmax=2,
    linewidths=0.4,
    linecolor="white",
    annot=False,
    ax=ax,
    cbar_kws={"label": "Z-score"}
)
ax.set_title("Top 15 variable genes — z-score expression")
ax.set_xlabel("Sample")
ax.set_ylabel("Gene")
plt.tight_layout()
plt.show()

---
## 7. Saving figures

In [ ]:
# Save the last figure
fig.savefig("heatmap_top15.png", dpi=150, bbox_inches="tight")
print("Saved: heatmap_top15.png")

# In Colab, download the file to your computer:
# from google.colab import files
# files.download("heatmap_top15.png")

---
## Mini-project — in-class

Produce a **2×2 figure** summarising the dataset. Starter code is provided — fill in the blanks.

Panel layout:
- **Top-left:** histogram of log2 expression values (all samples)
- **Top-right:** scatter plot of ctrl_1 vs treat_1
- **Bottom-left:** horizontal bar chart of top 10 genes by mean expression
- **Bottom-right:** text annotation (title card for the figure)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# --- Top-left: histogram ---
all_vals = df[sample_names].values.flatten()
axes[0, 0].hist(all_vals, bins=25, color="steelblue", edgecolor="white")
axes[0, 0].set_title("Distribution of all expression values")
axes[0, 0].set_xlabel("Expression (log2)")
axes[0, 0].set_ylabel("Frequency")

# --- Top-right: scatter ctrl_1 vs treat_1 ---
axes[0, 1].scatter(df["ctrl_1"], df["treat_1"], alpha=0.7, color="coral", edgecolors="white")
axes[0, 1].set_xlabel("ctrl_1 (log2)")
axes[0, 1].set_ylabel("treat_1 (log2)")
axes[0, 1].set_title("ctrl_1 vs treat_1")
# Add diagonal
lim_vals = [df[["ctrl_1", "treat_1"]].min().min() - 0.5,
            df[["ctrl_1", "treat_1"]].max().max() + 0.5]
axes[0, 1].plot(lim_vals, lim_vals, "k--", alpha=0.4)

# --- Bottom-left: bar chart top 10 ---
top10_bm = df.nlargest(10, "mean_ctrl").sort_values("mean_ctrl")
axes[1, 0].barh(top10_bm.index, top10_bm["mean_ctrl"], color="mediumpurple")
axes[1, 0].set_xlabel("Mean expression in control")
axes[1, 0].set_title("Top 10 expressed genes (control)")

# --- Bottom-right: text summary ---
axes[1, 1].axis("off")
summary_text = (
    "Dataset summary\n"
    "───────────────\n"
    f"Genes:   {len(df)}\n"
    f"Samples: {len(sample_names)}\n"
    f"Upregulated (log2FC>1): {(df['log2fc']>1).sum()}\n"
    f"Down (log2FC<-1):       {(df['log2fc']<-1).sum()}\n"
    f"Unchanged:              {((df['log2fc'].abs()<=1)).sum()}"
)
axes[1, 1].text(0.1, 0.5, summary_text, transform=axes[1, 1].transAxes,
                fontsize=12, verticalalignment="center", fontfamily="monospace")

plt.suptitle("Week 1 Mini-project — Expression Data Overview", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap in a separate cell (clustermap needs its own figure)
top15 = df.nlargest(15, "variance").index
heat_z = df.loc[top15, sample_names].apply(stats.zscore, axis=1)

cg = sns.clustermap(
    heat_z,
    cmap="RdBu_r", vmin=-2, vmax=2,
    figsize=(7, 7),
    linewidths=0.4,
    col_cluster=False,   # keep sample order
    cbar_pos=(0.02, 0.8, 0.03, 0.18)
)
cg.fig.suptitle("Clustered heatmap — top 15 variable genes", y=1.02)
plt.show()

---
## Exercises

### Exercise 1 — Volcano plot

A **volcano plot** is one of the most common plots in genomics. It shows:
- x-axis: log2 fold-change
- y-axis: −log10(p-value)

Genes in the top-right are significantly upregulated; top-left are significantly downregulated.

Create a volcano plot using `df['log2fc']` and `df['pvalue']`. Colour points that meet **both** criteria (|log2fc| > 1 and pvalue < 0.05) in red.

In [ ]:
# Exercise 1 — Volcano plot
# Hint: -log10(pvalue) amplifies small p-values into large positive numbers
# np.log10(df['pvalue']) — remember the minus sign!


### Exercise 2 — `sns.clustermap`

Use `sns.clustermap` on the **full** dataset (all 20 genes) with both row AND column clustering enabled (`row_cluster=True, col_cluster=True`).

Does the column clustering correctly separate control from treated samples? Do the upregulated genes (GENE_001–GENE_005) cluster together?

In [ ]:
# Exercise 2 — clustermap with both dimensions clustered


---
## Summary

| Plot type | Best for | Code |
|-----------|----------|------|
| Line plot | Trends across genes/time | `ax.plot()` |
| Bar chart | Rankings, counts | `ax.bar()` / `ax.barh()` |
| Histogram | Distributions | `ax.hist()` |
| Scatter plot | Correlations, volcano | `ax.scatter()` |
| Boxplot | Per-group distributions | `sns.boxplot()` |
| Heatmap | Matrix data (expression) | `sns.heatmap()` |
| Clustermap | Heatmap + clustering | `sns.clustermap()` |

**Week 1 complete!** You can now write Python programs to load, analyse, and visualise biological data. Week 2 will add statistical rigour to these analyses.